In [3]:
import os
import configparser
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr

# Baixa o conjunto de dados do Kaggle. Ele virá como um arquivo .zip
# A flag -p especifica onde salvar; a flag --unzip descompacta automaticamente.
# O ponto '.' significa "salvar e descompactar na pasta de dados atual"
print("Baixando e descompactando os dados do Kaggle...")
!kaggle datasets download -d cprete/covid19-open-datasets-for-brazil -p dados --unzip
print("Download concluído.")

# SESSÃO SPARK
caminho_driver_jdbc = os.path.abspath("../drivers/mssql-jdbc-13.2.1.jre11.jar")
spark = SparkSession.builder \
    .appName("ETL") \
    .config("spark.driver.extraClassPath", caminho_driver_jdbc) \
    .getOrCreate()

# EXTRAÇÃO
caminho_csv = 'dados/SARS_01-06/SRAG_01-06.csv'
print("Iniciando a leitura do CSV...")
df_bruto = spark.read \
    .option("header", "true") \
    .option("sep", ";") \
    .option("encoding", "iso-8859-1") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(caminho_csv)

# TRANSFORMAÇÃO
print("\nIniciando a transformação...")

df_selecionado = df_bruto.select(
    "DT_NOTIFIC", "SG_UF_NOT", "CS_SEXO", "DT_NASC", "HOSPITAL", 
    "UTI", "CLASSI_FIN", "EVOLUCAO"
)

# Usando try_cast para sobreviver a qualquer dado corrompido
df_transformado = df_selecionado \
    .withColumn("data_notificacao", expr("try_cast(trim(DT_NOTIFIC) as date)")) \
    .withColumn("data_nascimento", expr("try_cast(trim(DT_NASC) as date)")) \
    .select(
        "data_notificacao",
        col("SG_UF_NOT").alias("uf_notificacao"),
        col("CS_SEXO").alias("sexo"),
        "data_nascimento",
        col("HOSPITAL").cast("int").alias("foi_hospitalizado"),
        col("UTI").cast("int").alias("foi_para_uti"),
        col("CLASSI_FIN").cast("int").alias("classificacao_final"),
        col("EVOLUCAO").cast("int").alias("evolucao_caso")
    )

# FILTRO DE QUALIDADE: Remove linhas onde a data de notificação não pôde ser convertida
df_para_carga = df_transformado.na.drop(subset=["data_notificacao"])



# Lendo as configurações do arquivo .ini com as credenciais
config = configparser.ConfigParser()
config.read('config.ini')

db_host = config['database']['host']
db_name = config['database']['database_name']
db_user = config['database']['user']
db_password = config['database']['password']

#conectando ao banco
print("\nIniciando a carga dos dados para o SQL Server...")
jdbc_url = f"jdbc:sqlserver://{db_host};databaseName={db_name};encrypt=true;trustServerCertificate=true"
tabela_destino = "dbo.casos_srag_tratados"
propriedades_conexao = {
    "user": db_user,
    "password": db_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}
try:
    df_para_carga.write.jdbc(url=jdbc_url, table=tabela_destino, mode="overwrite", properties=propriedades_conexao)
    print(f"\n\n\033[1;32mETL COMPLETO! Os dados foram carregados na tabela '{tabela_destino}'.\033[0m")

except Exception as e:
    print(f"\nERRO durante a carga para o SQL Server: {e}")
finally:
    print("Encerrando a sessão Spark.")
    spark.stop()

Baixando e descompactando os dados do Kaggle...
Dataset URL: https://www.kaggle.com/datasets/cprete/covid19-open-datasets-for-brazil
License(s): DbCL-1.0

Download concluído.
Iniciando a leitura do CSV...



  0%|          | 0.00/139M [00:00<?, ?B/s]
100%|##########| 139M/139M [00:00<00:00, 1.83GB/s]



Iniciando a transformação...

Iniciando a carga dos dados para o SQL Server...


ETL COMPLETO! Os dados foram carregados na tabela 'dbo.casos_srag_tratados'.
Encerrando a sessão Spark.
